In [1]:
!pip -q install pystac-client planetary-computer rasterio geopandas shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 5.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os
import pandas as pd
import geopandas as gpd

drive.mount("/content/drive")

project_path = "/content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics"
processed_path = os.path.join(
    project_path,
    "data",
    "processed"
)

master_csv = os.path.join(
    processed_path,
    "hyderabad_ward_master_2011.csv"
)

master_geojson = os.path.join(
    processed_path,
    "hyderabad_ward_master_2011.geojson"
)

master_df = pd.read_csv(master_csv)
master_gdf = gpd.read_file(master_geojson)

print("CSV shape:", master_df.shape)
print("GeoJSON shape:", master_gdf.shape)
print("CRS:", master_gdf.crs)
print("Unique wards:", master_gdf["ward_id"].nunique())

Mounted at /content/drive
CSV shape: (150, 8)
GeoJSON shape: (150, 9)
CRS: EPSG:4326
Unique wards: 150


In [3]:
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

print("Connected to Microsoft Planetary Computer")

Connected to Microsoft Planetary Computer


In [4]:
from shapely.geometry import box

aoi = box(
    78.25, 17.20,
    78.65, 17.65
)

print("AOI created")
print("Bounds:", aoi.bounds)

AOI created
Bounds: (78.25, 17.2, 78.65, 17.65)


In [5]:
search = catalog.search(
    collections=["landsat-c2-l2"],
    intersects=aoi.__geo_interface__,
    datetime="2025-01-01/2025-12-31",
    query={"eo:cloud_cover": {"lt": 30}}
)

landsat_items = list(search.items())

print("Landsat scenes found:", len(landsat_items))

for item in landsat_items:
    print(
        item.datetime.date(),
        "|",
        item.id,
        "| Cloud:",
        item.properties.get("eo:cloud_cover")
    )

Landsat scenes found: 23
2025-12-18 | LC09_L2SP_144048_20251218_02_T1 | Cloud: 0.26
2025-12-10 | LC08_L2SP_144048_20251210_02_T1 | Cloud: 0.14
2025-12-02 | LC09_L2SP_144048_20251202_02_T1 | Cloud: 7.7
2025-11-16 | LC09_L2SP_144048_20251116_02_T1 | Cloud: 0.63
2025-11-08 | LC08_L2SP_144048_20251108_02_T1 | Cloud: 0.04
2025-10-31 | LC09_L2SP_144048_20251031_02_T1 | Cloud: 7.6
2025-09-29 | LC09_L2SP_144048_20250929_02_T1 | Cloud: 22.02
2025-06-01 | LC08_L2SP_144048_20250601_02_T1 | Cloud: 14.65
2025-05-08 | LC09_L2SP_144048_20250508_02_T1 | Cloud: 10.43
2025-04-30 | LC08_L2SP_144048_20250430_02_T1 | Cloud: 1.49
2025-04-22 | LC09_L2SP_144048_20250422_02_T1 | Cloud: 0.19
2025-04-14 | LC08_L2SP_144048_20250414_02_T1 | Cloud: 15.89
2025-04-06 | LC09_L2SP_144048_20250406_02_T1 | Cloud: 4.31
2025-03-29 | LC08_L2SP_144048_20250329_02_T1 | Cloud: 0.06
2025-03-21 | LC09_L2SP_144048_20250321_02_T1 | Cloud: 1.37
2025-03-13 | LC08_L2SP_144048_20250313_02_T1 | Cloud: 0.04
2025-03-05 | LC09_L2SP_144048

In [6]:
hot_season_items = [
    item for item in landsat_items
    if item.datetime.month in [3, 4]
    and item.properties.get("eo:cloud_cover", 100) < 10
]

hot_season_items = sorted(
    hot_season_items,
    key=lambda x: x.datetime
)

print("Selected Landsat scenes:", len(hot_season_items))

for item in hot_season_items:
    print(
        item.datetime.date(),
        "|",
        item.id,
        "| Cloud:",
        item.properties.get("eo:cloud_cover")
    )

Selected Landsat scenes: 7
2025-03-05 | LC09_L2SP_144048_20250305_02_T1 | Cloud: 6.52
2025-03-13 | LC08_L2SP_144048_20250313_02_T1 | Cloud: 0.04
2025-03-21 | LC09_L2SP_144048_20250321_02_T1 | Cloud: 1.37
2025-03-29 | LC08_L2SP_144048_20250329_02_T1 | Cloud: 0.06
2025-04-06 | LC09_L2SP_144048_20250406_02_T1 | Cloud: 4.31
2025-04-22 | LC09_L2SP_144048_20250422_02_T1 | Cloud: 0.19
2025-04-30 | LC08_L2SP_144048_20250430_02_T1 | Cloud: 1.49


In [7]:
item = hot_season_items[0]

print("Scene:", item.id)
print("\nAvailable assets:\n")

for name, asset in item.assets.items():
    print(name, "->", asset.title)

Scene: LC09_L2SP_144048_20250305_02_T1

Available assets:

qa -> Surface Temperature Quality Assessment Band
ang -> Angle Coefficients File
red -> Red Band
blue -> Blue Band
drad -> Downwelled Radiance Band
emis -> Emissivity Band
emsd -> Emissivity Standard Deviation Band
trad -> Thermal Radiance Band
urad -> Upwelled Radiance Band
atran -> Atmospheric Transmittance Band
cdist -> Cloud Distance Band
green -> Green Band
nir08 -> Near Infrared Band 0.8
lwir11 -> Surface Temperature Band
swir16 -> Short-wave Infrared Band 1.6
swir22 -> Short-wave Infrared Band 2.2
coastal -> Coastal/Aerosol Band
mtl.txt -> Product Metadata File (txt)
mtl.xml -> Product Metadata File (xml)
mtl.json -> Product Metadata File (json)
qa_pixel -> Pixel Quality Assessment Band
qa_radsat -> Radiometric Saturation and Terrain Occlusion Quality Assessment Band
qa_aerosol -> Aerosol Quality Assessment Band
tilejson -> TileJSON with default rendering
rendered_preview -> Rendered preview


In [8]:
import rasterio

item = hot_season_items[0]

for band_name in ["lwir11", "qa_pixel"]:

    asset = item.assets[band_name]

    with rasterio.open(asset.href) as src:

        print("\nBand:", band_name)
        print("CRS:", src.crs)
        print("Resolution:", src.res)
        print("Shape:", src.height, src.width)
        print("Dtype:", src.dtypes[0])
        print("Nodata:", src.nodata)
        print("Scale:", src.scales[0])
        print("Offset:", src.offsets[0])


Band: lwir11
CRS: EPSG:32644
Resolution: (30.0, 30.0)
Shape: 7821 7681
Dtype: uint16
Nodata: 0.0
Scale: 1.0
Offset: 0.0

Band: qa_pixel
CRS: EPSG:32644
Resolution: (30.0, 30.0)
Shape: 7821 7681
Dtype: uint16
Nodata: None
Scale: 1.0
Offset: 0.0


In [9]:
import numpy as np
import rasterio
from rasterio.mask import mask

item = hot_season_items[0]

ward_id = 1

ward_geom = master_gdf.loc[
    master_gdf["ward_id"] == ward_id,
    "geometry"
].iloc[0]

with rasterio.open(item.assets["lwir11"].href) as src:

    ward_gdf = gpd.GeoDataFrame(
        {"ward_id": [ward_id]},
        geometry=[ward_geom],
        crs=master_gdf.crs
    ).to_crs(src.crs)

    geometry = [
        ward_gdf.geometry.iloc[0].__geo_interface__
    ]

    lst_dn, lst_transform = mask(
        src,
        geometry,
        crop=True
    )

    scene_crs = src.crs

with rasterio.open(item.assets["qa_pixel"].href) as src:

    qa_pixel, qa_transform = mask(
        src,
        geometry,
        crop=True
    )

lst_dn = lst_dn[0]
qa_pixel = qa_pixel[0]

valid_qa = (
    ((qa_pixel & (1 << 0)) == 0) &
    ((qa_pixel & (1 << 1)) == 0) &
    ((qa_pixel & (1 << 2)) == 0) &
    ((qa_pixel & (1 << 3)) == 0) &
    ((qa_pixel & (1 << 4)) == 0) &
    ((qa_pixel & (1 << 5)) == 0)
)

valid_lst = (
    (lst_dn > 0) &
    (lst_dn != 65535)
)

valid = valid_lst & valid_qa

lst_kelvin = (
    lst_dn[valid] * 0.00341802 + 149.0
)

lst_celsius = lst_kelvin - 273.15

lst_celsius = lst_celsius[
    (lst_celsius > -20) &
    (lst_celsius < 80)
]

print("Scene:", item.id)
print("Ward:", ward_id)
print("Scene CRS:", scene_crs)
print("Valid pixels:", len(lst_celsius))

print("\nLST statistics:")
print("Mean:", np.mean(lst_celsius))
print("Median:", np.median(lst_celsius))
print("Min:", np.min(lst_celsius))
print("Max:", np.max(lst_celsius))


Scene: LC09_L2SP_144048_20250305_02_T1
Ward: 1
Scene CRS: EPSG:32644
Valid pixels: 9953

LST statistics:
Mean: 37.766760701328266
Median: 37.806041660000005
Min: 28.389396560000023
Max: 43.08688256000005


In [10]:
from rasterio.mask import mask
from rasterio.errors import WindowError

lst_results = []

for scene_num, item in enumerate(hot_season_items, start=1):

    print(
        f"Processing {scene_num}/{len(hot_season_items)}:",
        item.id
    )

    with rasterio.open(item.assets["lwir11"].href) as lst_src:

        scene_crs = lst_src.crs

        for _, ward_row in master_gdf.iterrows():

            ward_id = int(ward_row["ward_id"])

            ward_gdf = gpd.GeoDataFrame(
                {"ward_id": [ward_id]},
                geometry=[ward_row.geometry],
                crs=master_gdf.crs
            ).to_crs(scene_crs)

            geometry = [
                ward_gdf.geometry.iloc[0].__geo_interface__
            ]

            try:
                lst_dn, _ = mask(
                    lst_src,
                    geometry,
                    crop=True
                )
            except (ValueError, WindowError):
                continue

            lst_dn = lst_dn[0]

            with rasterio.open(item.assets["qa_pixel"].href) as qa_src:

                try:
                    qa_pixel, _ = mask(
                        qa_src,
                        geometry,
                        crop=True
                    )
                except (ValueError, WindowError):
                    continue

            qa_pixel = qa_pixel[0]

            valid_qa = (
                ((qa_pixel & (1 << 0)) == 0) &
                ((qa_pixel & (1 << 1)) == 0) &
                ((qa_pixel & (1 << 2)) == 0) &
                ((qa_pixel & (1 << 3)) == 0) &
                ((qa_pixel & (1 << 4)) == 0) &
                ((qa_pixel & (1 << 5)) == 0)
            )

            valid_lst = (
                (lst_dn > 0) &
                (lst_dn != 65535)
            )

            valid = valid_lst & valid_qa

            if not np.any(valid):
                continue

            lst_kelvin = (
                lst_dn[valid] * 0.00341802 + 149.0
            )

            lst_celsius = lst_kelvin - 273.15

            lst_celsius = lst_celsius[
                (lst_celsius > -20) &
                (lst_celsius < 80)
            ]

            if len(lst_celsius) == 0:
                continue

            lst_results.append({
                "ward_id": ward_id,
                "date": item.datetime.date(),
                "scene_id": item.id,
                "lst_mean_c": float(np.mean(lst_celsius)),
                "lst_median_c": float(np.median(lst_celsius)),
                "lst_min_c": float(np.min(lst_celsius)),
                "lst_max_c": float(np.max(lst_celsius)),
                "valid_pixels": int(len(lst_celsius))
            })

lst_scene_df = pd.DataFrame(lst_results)

print("\nProcessing complete.")
print("Rows:", len(lst_scene_df))
print("Unique wards:", lst_scene_df["ward_id"].nunique())
print("Unique scenes:", lst_scene_df["scene_id"].nunique())
print("Missing median LST:", lst_scene_df["lst_median_c"].isna().sum())

Processing 1/7: LC09_L2SP_144048_20250305_02_T1
Processing 2/7: LC08_L2SP_144048_20250313_02_T1
Processing 3/7: LC09_L2SP_144048_20250321_02_T1
Processing 4/7: LC08_L2SP_144048_20250329_02_T1
Processing 5/7: LC09_L2SP_144048_20250406_02_T1
Processing 6/7: LC09_L2SP_144048_20250422_02_T1
Processing 7/7: LC08_L2SP_144048_20250430_02_T1

Processing complete.
Rows: 1050
Unique wards: 150
Unique scenes: 7
Missing median LST: 0


In [11]:
ward_lst_2025 = (
    lst_scene_df
    .groupby("ward_id", as_index=False)
    .agg(
        lst_2025_c=("lst_median_c", "median"),
        lst_mean_across_scenes_c=("lst_mean_c", "mean"),
        lst_min_observed_c=("lst_min_c", "min"),
        lst_max_observed_c=("lst_max_c", "max"),
        observations_lst=("date", "count"),
        total_valid_pixels_lst=("valid_pixels", "sum")
    )
)

print("Rows:", len(ward_lst_2025))
print("Unique wards:", ward_lst_2025["ward_id"].nunique())
print("Missing LST:", ward_lst_2025["lst_2025_c"].isna().sum())

print("\nLST summary:")
print(
    ward_lst_2025["lst_2025_c"].describe()
)

print("\nFirst 10 wards:")
print(
    ward_lst_2025.head(10)
)

Rows: 150
Unique wards: 150
Missing LST: 0

LST summary:
count    150.000000
mean      38.888437
std        0.944894
min       33.506173
25%       38.265765
50%       38.804103
75%       39.481726
max       41.719675
Name: lst_2025_c, dtype: float64

First 10 wards:
   ward_id  lst_2025_c  lst_mean_across_scenes_c  lst_min_observed_c  \
0        1   37.963271                 38.635793           28.389397   
1        2   37.864148                 38.601001           32.637995   
2        3   39.450109                 39.841823           28.430413   
3        4   38.506736                 39.497064           34.080400   
4        5   39.815837                 40.384496           33.451484   
5        6   38.510154                 39.296597           27.962144   
6        7   38.264056                 38.678876           27.886948   
7        8   38.238421                 38.500473           29.777113   
8        9   38.722071                 39.082390           29.794203   
9       10   

In [12]:
heat_master = master_gdf.merge(
    ward_lst_2025,
    on="ward_id",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(heat_master))
print("Columns:", len(heat_master.columns))
print("Unique wards:", heat_master["ward_id"].nunique())
print("Missing LST:", heat_master["lst_2025_c"].isna().sum())

print("\nColumns:")
print(heat_master.columns.tolist())

Rows: 150
Columns: 15
Unique wards: 150
Missing LST: 0

Columns:
['ward_id', 'ward_name', 'circle', 'zone', 'area_sq_km', 'population_2011', 'households_2011', 'population_density_2011', 'geometry', 'lst_2025_c', 'lst_mean_across_scenes_c', 'lst_min_observed_c', 'lst_max_observed_c', 'observations', 'total_valid_pixels']


In [13]:
lst_master_geojson = os.path.join(
    processed_path,
    "hyderabad_ward_heat_master_2025.geojson"
)

lst_master_csv = os.path.join(
    processed_path,
    "hyderabad_ward_heat_master_2025.csv"
)

heat_master.to_file(
    lst_master_geojson,
    driver="GeoJSON"
)

heat_master.drop(
    columns="geometry"
).to_csv(
    lst_master_csv,
    index=False
)

print("Saved:")
print(lst_master_geojson)
print(lst_master_csv)

Saved:
/content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/hyderabad_ward_heat_master_2025.geojson
/content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/hyderabad_ward_heat_master_2025.csv


In [14]:
s2_search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=aoi.__geo_interface__,
    datetime="2025-03-01/2025-04-30",
    query={"eo:cloud_cover": {"lt": 20}}
)

s2_items = list(s2_search.items())

print("Sentinel-2 scenes found:", len(s2_items))

for item in s2_items:
    print(
        item.datetime.date(),
        "|",
        item.id,
        "| Cloud:",
        item.properties.get("eo:cloud_cover")
    )

Sentinel-2 scenes found: 23
2025-04-24 | S2A_MSIL2A_20250424T051241_R019_T44QKE_20250424T102418 | Cloud: 15.219943
2025-04-24 | S2A_MSIL2A_20250424T051241_R019_T43QHV_20250424T102418 | Cloud: 10.939203
2025-04-22 | S2C_MSIL2A_20250422T050711_R019_T44QKE_20250422T102513 | Cloud: 0.002857
2025-04-22 | S2C_MSIL2A_20250422T050711_R019_T43QHV_20250422T102513 | Cloud: 0.003221
2025-04-17 | S2B_MSIL2A_20250417T050659_R019_T44QKE_20250417T074435 | Cloud: 14.054081
2025-04-17 | S2B_MSIL2A_20250417T050659_R019_T43QHV_20250417T074435 | Cloud: 0.7688
2025-04-12 | S2C_MSIL2A_20250412T050711_R019_T44QKE_20250412T102015 | Cloud: 5.257259
2025-04-12 | S2C_MSIL2A_20250412T050711_R019_T43QHV_20250412T102015 | Cloud: 5.576475
2025-04-07 | S2B_MSIL2A_20250407T050649_R019_T44QKE_20250407T074451 | Cloud: 0.004124
2025-04-07 | S2B_MSIL2A_20250407T050649_R019_T43QHV_20250407T074451 | Cloud: 0.00369
2025-04-04 | S2A_MSIL2A_20250404T051241_R019_T44QKE_20250404T105803 | Cloud: 15.755887
2025-03-28 | S2B_MSIL2A_2

In [15]:
s2_selected = [
    item for item in s2_items
    if item.datetime.month in [3, 4]
    and item.properties.get("eo:cloud_cover", 100) < 10
]

s2_selected = sorted(
    s2_selected,
    key=lambda x: (x.datetime, x.id)
)

print("Selected Sentinel-2 scenes:", len(s2_selected))

for item in s2_selected:
    print(
        item.datetime.date(),
        "|",
        item.id,
        "| Cloud:",
        item.properties.get("eo:cloud_cover")
    )

Selected Sentinel-2 scenes: 19
2025-03-03 | S2C_MSIL2A_20250303T050741_R019_T43QHV_20250303T103811 | Cloud: 8.789667
2025-03-03 | S2C_MSIL2A_20250303T050741_R019_T44QKE_20250303T103811 | Cloud: 4.518416
2025-03-08 | S2B_MSIL2A_20250308T050659_R019_T43QHV_20250308T072151 | Cloud: 0.002363
2025-03-08 | S2B_MSIL2A_20250308T050659_R019_T44QKE_20250308T072151 | Cloud: 0.002236
2025-03-13 | S2C_MSIL2A_20250313T050711_R019_T43QHV_20250313T102208 | Cloud: 0.00134
2025-03-13 | S2C_MSIL2A_20250313T050711_R019_T44QKE_20250313T102208 | Cloud: 0.001666
2025-03-15 | S2A_MSIL2A_20250315T051241_R019_T43QHV_20250315T113317 | Cloud: 0.0013
2025-03-15 | S2A_MSIL2A_20250315T051241_R019_T44QKE_20250315T113317 | Cloud: 0.001261
2025-03-18 | S2B_MSIL2A_20250318T050659_R019_T43QHV_20250318T080831 | Cloud: 0.001601
2025-03-18 | S2B_MSIL2A_20250318T050659_R019_T44QKE_20250318T080831 | Cloud: 0.723279
2025-03-28 | S2B_MSIL2A_20250328T050659_R019_T43QHV_20250328T072008 | Cloud: 0.000894
2025-03-28 | S2B_MSIL2A_20

In [16]:
for tile in ["T43QHV", "T44QKE"]:

    item = next(
        x for x in s2_selected
        if tile in x.id
    )

    print("\nTile:", tile)
    print("Scene:", item.id)
    print("\nAvailable assets:")

    for name, asset in item.assets.items():
        print(name, "->", asset.title)


Tile: T43QHV
Scene: S2C_MSIL2A_20250303T050741_R019_T43QHV_20250303T103811

Available assets:
AOT -> Aerosol optical thickness (AOT)
B01 -> Band 1 - Coastal aerosol - 60m
B02 -> Band 2 - Blue - 10m
B03 -> Band 3 - Green - 10m
B04 -> Band 4 - Red - 10m
B05 -> Band 5 - Vegetation red edge 1 - 20m
B06 -> Band 6 - Vegetation red edge 2 - 20m
B07 -> Band 7 - Vegetation red edge 3 - 20m
B08 -> Band 8 - NIR - 10m
B09 -> Band 9 - Water vapor - 60m
B11 -> Band 11 - SWIR (1.6) - 20m
B12 -> Band 12 - SWIR (2.2) - 20m
B8A -> Band 8A - Vegetation red edge 4 - 20m
SCL -> Scene classfication map (SCL)
WVP -> Water vapour (WVP)
visual -> True color image
safe-manifest -> SAFE manifest
granule-metadata -> Granule metadata
inspire-metadata -> INSPIRE metadata
product-metadata -> Product metadata
datastrip-metadata -> Datastrip metadata
tilejson -> TileJSON with default rendering
rendered_preview -> Rendered preview

Tile: T44QKE
Scene: S2C_MSIL2A_20250303T050741_R019_T44QKE_20250303T103811

Available a

In [17]:
import rasterio

for tile in ["T43QHV", "T44QKE"]:

    item = next(
        x for x in s2_selected
        if tile in x.id
    )

    print("\nTile:", tile)
    print("Scene:", item.id)

    for band_name in ["B04", "B08", "SCL"]:

        with rasterio.open(item.assets[band_name].href) as src:

            print(
                band_name,
                "| CRS:", src.crs,
                "| Resolution:", src.res,
                "| Shape:", (src.height, src.width),
                "| Dtype:", src.dtypes[0],
                "| Nodata:", src.nodata
            )



Tile: T43QHV
Scene: S2C_MSIL2A_20250303T050741_R019_T43QHV_20250303T103811
B04 | CRS: EPSG:32643 | Resolution: (10.0, 10.0) | Shape: (10980, 10980) | Dtype: uint16 | Nodata: 0.0
B08 | CRS: EPSG:32643 | Resolution: (10.0, 10.0) | Shape: (10980, 10980) | Dtype: uint16 | Nodata: 0.0
SCL | CRS: EPSG:32643 | Resolution: (20.0, 20.0) | Shape: (5490, 5490) | Dtype: uint8 | Nodata: 0.0

Tile: T44QKE
Scene: S2C_MSIL2A_20250303T050741_R019_T44QKE_20250303T103811
B04 | CRS: EPSG:32644 | Resolution: (10.0, 10.0) | Shape: (10980, 10980) | Dtype: uint16 | Nodata: 0.0
B08 | CRS: EPSG:32644 | Resolution: (10.0, 10.0) | Shape: (10980, 10980) | Dtype: uint16 | Nodata: 0.0
SCL | CRS: EPSG:32644 | Resolution: (20.0, 20.0) | Shape: (5490, 5490) | Dtype: uint8 | Nodata: 0.0


In [18]:
from rasterio.mask import mask
from rasterio.enums import Resampling
from rasterio.warp import reproject

item = next(
    x for x in s2_selected
    if "T43QHV" in x.id
)

ward_id = 1

ward_geom = master_gdf.loc[
    master_gdf["ward_id"] == ward_id,
    "geometry"
].iloc[0]

with rasterio.open(item.assets["B04"].href) as red_src:

    scene_crs = red_src.crs

    ward_gdf = gpd.GeoDataFrame(
        {"ward_id": [ward_id]},
        geometry=[ward_geom],
        crs=master_gdf.crs
    ).to_crs(scene_crs)

    geometry = [
        ward_gdf.geometry.iloc[0].__geo_interface__
    ]

    red, red_transform = mask(
        red_src,
        geometry,
        crop=True
    )

    red_crs = red_src.crs

with rasterio.open(item.assets["B08"].href) as nir_src:

    nir, nir_transform = mask(
        nir_src,
        geometry,
        crop=True
    )

with rasterio.open(item.assets["SCL"].href) as scl_src:

    scl_20m, scl_transform = mask(
        scl_src,
        geometry,
        crop=True
    )

    scl_crs = scl_src.crs

red = red[0].astype("float32")
nir = nir[0].astype("float32")
scl_20m = scl_20m[0]

scl_10m = np.empty(
    red.shape,
    dtype=np.uint8
)

reproject(
    source=scl_20m,
    destination=scl_10m,
    src_transform=scl_transform,
    src_crs=scl_crs,
    dst_transform=red_transform,
    dst_crs=red_crs,
    resampling=Resampling.nearest
)

valid_reflectance = (
    (red > 0) &
    (nir > 0) &
    np.isfinite(red) &
    np.isfinite(nir) &
    ((nir + red) != 0)
)

valid_scl = np.isin(
    scl_10m,
    [4, 5]
)

valid = (
    valid_reflectance &
    valid_scl
)

ndvi = (
    (nir[valid] - red[valid]) /
    (nir[valid] + red[valid])
)

ndvi = ndvi[
    np.isfinite(ndvi) &
    (ndvi >= -1) &
    (ndvi <= 1)
]

print("Scene:", item.id)
print("Tile:", "T43QHV")
print("Ward:", ward_id)
print("Scene CRS:", scene_crs)
print("Valid NDVI pixels:", len(ndvi))

print("\nNDVI statistics:")
print("Mean:", np.mean(ndvi))
print("Median:", np.median(ndvi))
print("Min:", np.min(ndvi))
print("Max:", np.max(ndvi))

Scene: S2C_MSIL2A_20250303T050741_R019_T43QHV_20250303T103811
Tile: T43QHV
Ward: 1
Scene CRS: EPSG:32643
Valid NDVI pixels: 87447

NDVI statistics:
Mean: 0.15880251
Median: 0.1383855
Min: -0.057846814
Max: 0.6730514


In [19]:
item = next(
    x for x in s2_selected
    if "T44QKE" in x.id
)

ward_id = 150

ward_geom = master_gdf.loc[
    master_gdf["ward_id"] == ward_id,
    "geometry"
].iloc[0]

with rasterio.open(item.assets["B04"].href) as red_src:

    scene_crs = red_src.crs

    ward_gdf = gpd.GeoDataFrame(
        {"ward_id": [ward_id]},
        geometry=[ward_geom],
        crs=master_gdf.crs
    ).to_crs(scene_crs)

    geometry = [
        ward_gdf.geometry.iloc[0].__geo_interface__
    ]

    red, red_transform = mask(
        red_src,
        geometry,
        crop=True
    )

    red_crs = red_src.crs

with rasterio.open(item.assets["B08"].href) as nir_src:

    nir, nir_transform = mask(
        nir_src,
        geometry,
        crop=True
    )

with rasterio.open(item.assets["SCL"].href) as scl_src:

    scl_20m, scl_transform = mask(
        scl_src,
        geometry,
        crop=True
    )

    scl_crs = scl_src.crs

red = red[0].astype("float32")
nir = nir[0].astype("float32")
scl_20m = scl_20m[0]

scl_10m = np.empty(
    red.shape,
    dtype=np.uint8
)

reproject(
    source=scl_20m,
    destination=scl_10m,
    src_transform=scl_transform,
    src_crs=scl_crs,
    dst_transform=red_transform,
    dst_crs=red_crs,
    resampling=Resampling.nearest
)

valid_reflectance = (
    (red > 0) &
    (nir > 0) &
    np.isfinite(red) &
    np.isfinite(nir) &
    ((nir + red) != 0)
)

valid_scl = np.isin(
    scl_10m,
    [4, 5]
)

valid = (
    valid_reflectance &
    valid_scl
)

ndvi = (
    (nir[valid] - red[valid]) /
    (nir[valid] + red[valid])
)

ndvi = ndvi[
    np.isfinite(ndvi) &
    (ndvi >= -1) &
    (ndvi <= 1)
]

print("Scene:", item.id)
print("Tile:", "T44QKE")
print("Ward:", ward_id)
print("Scene CRS:", scene_crs)
print("Valid NDVI pixels:", len(ndvi))

print("\nNDVI statistics:")
print("Mean:", np.mean(ndvi))
print("Median:", np.median(ndvi))
print("Min:", np.min(ndvi))
print("Max:", np.max(ndvi))

Scene: S2C_MSIL2A_20250303T050741_R019_T44QKE_20250303T103811
Tile: T44QKE
Ward: 150
Scene CRS: EPSG:32644
Valid NDVI pixels: 25315

NDVI statistics:
Mean: 0.121876754
Median: 0.087638825
Min: -0.12850164
Max: 0.56385726


In [20]:
ndvi_results = []

for scene_num, item in enumerate(s2_selected, start=1):

    print(
        f"Processing {scene_num}/{len(s2_selected)}:",
        item.id
    )

    with rasterio.open(item.assets["B04"].href) as red_src:

        scene_crs = red_src.crs

        with rasterio.open(item.assets["B08"].href) as nir_src:
            with rasterio.open(item.assets["SCL"].href) as scl_src:

                for _, ward_row in master_gdf.iterrows():

                    ward_id = int(ward_row["ward_id"])

                    ward_gdf = gpd.GeoDataFrame(
                        {"ward_id": [ward_id]},
                        geometry=[ward_row.geometry],
                        crs=master_gdf.crs
                    ).to_crs(scene_crs)

                    geometry = [
                        ward_gdf.geometry.iloc[0].__geo_interface__
                    ]

                    try:

                        red, red_transform = mask(
                            red_src,
                            geometry,
                            crop=True
                        )

                        nir, _ = mask(
                            nir_src,
                            geometry,
                            crop=True
                        )

                        scl_20m, scl_transform = mask(
                            scl_src,
                            geometry,
                            crop=True
                        )

                    except (ValueError, WindowError):
                        continue

                    red = red[0].astype("float32")
                    nir = nir[0].astype("float32")
                    scl_20m = scl_20m[0]

                    scl_10m = np.empty(
                        red.shape,
                        dtype=np.uint8
                    )

                    reproject(
                        source=scl_20m,
                        destination=scl_10m,
                        src_transform=scl_transform,
                        src_crs=scl_src.crs,
                        dst_transform=red_transform,
                        dst_crs=red_src.crs,
                        resampling=Resampling.nearest
                    )

                    valid_reflectance = (
                        (red > 0) &
                        (nir > 0) &
                        np.isfinite(red) &
                        np.isfinite(nir) &
                        ((nir + red) != 0)
                    )

                    valid_scl = np.isin(
                        scl_10m,
                        [4, 5]
                    )

                    valid = (
                        valid_reflectance &
                        valid_scl
                    )

                    if not np.any(valid):
                        continue

                    ndvi = (
                        (nir[valid] - red[valid]) /
                        (nir[valid] + red[valid])
                    )

                    ndvi = ndvi[
                        np.isfinite(ndvi) &
                        (ndvi >= -1) &
                        (ndvi <= 1)
                    ]

                    if len(ndvi) == 0:
                        continue

                    ndvi_results.append({
                        "ward_id": ward_id,
                        "date": item.datetime.date(),
                        "scene_id": item.id,
                        "tile": (
                            "T43QHV"
                            if "T43QHV" in item.id
                            else "T44QKE"
                        ),
                        "ndvi_mean": float(np.mean(ndvi)),
                        "ndvi_median": float(np.median(ndvi)),
                        "ndvi_min": float(np.min(ndvi)),
                        "ndvi_max": float(np.max(ndvi)),
                        "valid_pixels": int(len(ndvi))
                    })

ndvi_scene_df = pd.DataFrame(ndvi_results)

print("\nProcessing complete.")
print("Rows:", len(ndvi_scene_df))
print("Unique wards:", ndvi_scene_df["ward_id"].nunique())
print("Unique scenes:", ndvi_scene_df["scene_id"].nunique())
print("Unique tiles:", ndvi_scene_df["tile"].nunique())
print("Missing median NDVI:", ndvi_scene_df["ndvi_median"].isna().sum())

Processing 1/19: S2C_MSIL2A_20250303T050741_R019_T43QHV_20250303T103811
Processing 2/19: S2C_MSIL2A_20250303T050741_R019_T44QKE_20250303T103811
Processing 3/19: S2B_MSIL2A_20250308T050659_R019_T43QHV_20250308T072151
Processing 4/19: S2B_MSIL2A_20250308T050659_R019_T44QKE_20250308T072151
Processing 5/19: S2C_MSIL2A_20250313T050711_R019_T43QHV_20250313T102208
Processing 6/19: S2C_MSIL2A_20250313T050711_R019_T44QKE_20250313T102208
Processing 7/19: S2A_MSIL2A_20250315T051241_R019_T43QHV_20250315T113317
Processing 8/19: S2A_MSIL2A_20250315T051241_R019_T44QKE_20250315T113317
Processing 9/19: S2B_MSIL2A_20250318T050659_R019_T43QHV_20250318T080831
Processing 10/19: S2B_MSIL2A_20250318T050659_R019_T44QKE_20250318T080831
Processing 11/19: S2B_MSIL2A_20250328T050659_R019_T43QHV_20250328T072008
Processing 12/19: S2B_MSIL2A_20250328T050659_R019_T44QKE_20250328T072008
Processing 13/19: S2B_MSIL2A_20250407T050649_R019_T43QHV_20250407T074451
Processing 14/19: S2B_MSIL2A_20250407T050649_R019_T44QKE_202

In [21]:
ward_ndvi_2025 = (
    ndvi_scene_df
    .groupby("ward_id", as_index=False)
    .agg(
        ndvi_2025=("ndvi_median", "median"),
        ndvi_mean_across_scenes=("ndvi_mean", "mean"),
        ndvi_min_observed=("ndvi_min", "min"),
        ndvi_max_observed=("ndvi_max", "max"),
        observations_ndvi=("date", "count"),
        total_valid_pixels_ndvi=("valid_pixels", "sum")
    )
)

print("Rows:", len(ward_ndvi_2025))
print("Unique wards:", ward_ndvi_2025["ward_id"].nunique())
print("Missing NDVI:", ward_ndvi_2025["ndvi_2025"].isna().sum())

print("\nNDVI summary:")
print(
    ward_ndvi_2025["ndvi_2025"].describe()
)

print("\nFirst 10 wards:")
print(
    ward_ndvi_2025.head(10)
)

Rows: 150
Unique wards: 150
Missing NDVI: 0

NDVI summary:
count    150.000000
mean       0.109875
std        0.032651
min        0.060739
25%        0.085765
50%        0.105400
75%        0.127925
max        0.286624
Name: ndvi_2025, dtype: float64

First 10 wards:
   ward_id  ndvi_2025  ndvi_mean_across_scenes  ndvi_min_observed  \
0        1   0.148417                 0.184520          -0.085381   
1        2   0.136789                 0.177950          -0.148428   
2        3   0.143996                 0.179490          -0.197085   
3        4   0.121354                 0.153536          -0.080963   
4        5   0.134192                 0.166182          -0.255331   
5        6   0.125300                 0.166300          -0.094758   
6        7   0.131655                 0.169900          -0.115616   
7        8   0.161975                 0.192077          -0.127463   
8        9   0.106715                 0.148320          -0.146413   
9       10   0.127327                 0.16

In [25]:
heat_master = heat_master.merge(
    ward_ndvi_2025,
    on="ward_id",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(heat_master))
print("Columns:", len(heat_master.columns))
print("Unique wards:", heat_master["ward_id"].nunique())
print("Missing LST:", heat_master["lst_2025_c"].isna().sum())
print("Missing NDVI:", heat_master["ndvi_2025"].isna().sum())

print("\nColumns:")
print(heat_master.columns.tolist())

Rows: 150
Columns: 33
Unique wards: 150
Missing LST: 0
Missing NDVI: 0

Columns:
['ward_id', 'ward_name', 'circle', 'zone', 'area_sq_km', 'population_2011', 'households_2011', 'population_density_2011', 'geometry', 'lst_2025_c', 'lst_mean_across_scenes_c', 'lst_min_observed_c', 'lst_max_observed_c', 'observations_x', 'total_valid_pixels_x', 'ndvi_2025_x', 'ndvi_mean_across_scenes_x', 'ndvi_min_observed_x', 'ndvi_max_observed_x', 'observations_y', 'total_valid_pixels_y', 'ndvi_2025_y', 'ndvi_mean_across_scenes_y', 'ndvi_min_observed_y', 'ndvi_max_observed_y', 'observations_lst', 'total_valid_pixels_lst', 'ndvi_2025', 'ndvi_mean_across_scenes', 'ndvi_min_observed', 'ndvi_max_observed', 'observations_ndvi', 'total_valid_pixels_ndvi']


In [26]:
ndvi_columns_to_drop = [
    col for col in heat_master.columns
    if col.startswith("ndvi_") and col != "ndvi_2025"
]

heat_master = heat_master.drop(
    columns=ndvi_columns_to_drop,
    errors="ignore"
)

observation_columns_to_drop = [
    col for col in heat_master.columns
    if col.startswith("observations_") and col != "observations_lst"
]

pixel_columns_to_drop = [
    col for col in heat_master.columns
    if col.startswith("total_valid_pixels_") and col != "total_valid_pixels_lst"
]

heat_master = heat_master.drop(
    columns=observation_columns_to_drop + pixel_columns_to_drop,
    errors="ignore"
)

print("Rows:", len(heat_master))
print("Columns:", len(heat_master.columns))
print("Unique wards:", heat_master["ward_id"].nunique())

print("\nNDVI columns:")
print([col for col in heat_master.columns if "ndvi" in col.lower()])

print("\nObservation columns:")
print([col for col in heat_master.columns if "observations" in col.lower()])

print("\nMissing values:")
print(
    heat_master[
        ["lst_2025_c", "ndvi_2025"]
    ].isna().sum()
)

Rows: 150
Columns: 16
Unique wards: 150

NDVI columns:
['ndvi_2025']

Observation columns:
['observations_lst']

Missing values:
lst_2025_c    0
ndvi_2025     0
dtype: int64


In [27]:
item = s2_selected[0]

print("Scene:", item.id)
print("Date:", item.datetime.date())
print()

for band in ["B08", "B11"]:
    asset = item.assets[band]

    print(band)
    print("Title:", asset.title)
    print("Href available:", bool(asset.href))

    with rasterio.open(asset.href) as src:
        print("CRS:", src.crs)
        print("Resolution:", src.res)
        print("Shape:", src.shape)
        print("Dtype:", src.dtypes[0])
        print("Nodata:", src.nodata)
        print()

Scene: S2C_MSIL2A_20250303T050741_R019_T43QHV_20250303T103811
Date: 2025-03-03

B08
Title: Band 8 - NIR - 10m
Href available: True
CRS: EPSG:32643
Resolution: (10.0, 10.0)
Shape: (10980, 10980)
Dtype: uint16
Nodata: 0.0

B11
Title: Band 11 - SWIR (1.6) - 20m
Href available: True
CRS: EPSG:32643
Resolution: (20.0, 20.0)
Shape: (5490, 5490)
Dtype: uint16
Nodata: 0.0



In [28]:
from rasterio.mask import mask
from rasterio.enums import Resampling
from rasterio.warp import reproject

test_item = s2_selected[0]
test_ward = master_gdf[master_gdf["ward_id"] == 1].iloc[0]

with rasterio.open(test_item.assets["B08"].href) as nir_src:
    with rasterio.open(test_item.assets["B11"].href) as swir_src:

        ward_gdf = gpd.GeoDataFrame(
            {"ward_id": [1]},
            geometry=[test_ward.geometry],
            crs=master_gdf.crs
        ).to_crs(swir_src.crs)

        geometry = [ward_gdf.geometry.iloc[0].__geo_interface__]

        nir_10m, nir_transform = mask(
            nir_src,
            geometry,
            crop=True
        )

        swir_20m, swir_transform = mask(
            swir_src,
            geometry,
            crop=True
        )

        nir_10m = nir_10m[0].astype("float32")
        swir_20m = swir_20m[0].astype("float32")

        nir_20m = np.empty(
            swir_20m.shape,
            dtype="float32"
        )

        reproject(
            source=nir_10m,
            destination=nir_20m,
            src_transform=nir_transform,
            src_crs=nir_src.crs,
            dst_transform=swir_transform,
            dst_crs=swir_src.crs,
            resampling=Resampling.bilinear
        )

        valid = (
            (nir_20m > 0) &
            (swir_20m > 0) &
            np.isfinite(nir_20m) &
            np.isfinite(swir_20m) &
            ((nir_20m + swir_20m) != 0)
        )

        ndbi = (
            (swir_20m[valid] - nir_20m[valid]) /
            (swir_20m[valid] + nir_20m[valid])
        )

        ndbi = ndbi[
            np.isfinite(ndbi) &
            (ndbi >= -1) &
            (ndbi <= 1)
        ]

print("Ward:", 1)
print("Scene:", test_item.id)
print("Valid NDBI pixels:", len(ndbi))
print("Mean NDBI:", float(np.mean(ndbi)))
print("Median NDBI:", float(np.median(ndbi)))
print("Minimum NDBI:", float(np.min(ndbi)))
print("Maximum NDBI:", float(np.max(ndbi)))

Ward: 1
Scene: S2C_MSIL2A_20250303T050741_R019_T43QHV_20250303T103811
Valid NDBI pixels: 22433
Mean NDBI: 0.04470907896757126
Median NDBI: 0.04483699053525925
Minimum NDBI: -0.3616865575313568
Maximum NDBI: 0.658730685710907


In [29]:
ndbi_results = []

for scene_num, item in enumerate(s2_selected, start=1):
    print(f"Processing {scene_num}/{len(s2_selected)}:", item.id)

    with rasterio.open(item.assets["B08"].href) as nir_src:
        with rasterio.open(item.assets["B11"].href) as swir_src:

            scene_crs = swir_src.crs

            for _, ward_row in master_gdf.iterrows():
                ward_id = int(ward_row["ward_id"])

                ward_gdf = gpd.GeoDataFrame(
                    {"ward_id": [ward_id]},
                    geometry=[ward_row.geometry],
                    crs=master_gdf.crs
                ).to_crs(scene_crs)

                geometry = [
                    ward_gdf.geometry.iloc[0].__geo_interface__
                ]

                try:
                    nir_10m, nir_transform = mask(
                        nir_src,
                        geometry,
                        crop=True
                    )

                    swir_20m, swir_transform = mask(
                        swir_src,
                        geometry,
                        crop=True
                    )

                except (ValueError, WindowError):
                    continue

                nir_10m = nir_10m[0].astype("float32")
                swir_20m = swir_20m[0].astype("float32")

                nir_20m = np.empty(
                    swir_20m.shape,
                    dtype="float32"
                )

                reproject(
                    source=nir_10m,
                    destination=nir_20m,
                    src_transform=nir_transform,
                    src_crs=nir_src.crs,
                    dst_transform=swir_transform,
                    dst_crs=swir_src.crs,
                    resampling=Resampling.bilinear
                )

                valid = (
                    (nir_20m > 0) &
                    (swir_20m > 0) &
                    np.isfinite(nir_20m) &
                    np.isfinite(swir_20m) &
                    ((nir_20m + swir_20m) != 0)
                )

                if not np.any(valid):
                    continue

                ndbi = (
                    (swir_20m[valid] - nir_20m[valid]) /
                    (swir_20m[valid] + nir_20m[valid])
                )

                ndbi = ndbi[
                    np.isfinite(ndbi) &
                    (ndbi >= -1) &
                    (ndbi <= 1)
                ]

                if len(ndbi) == 0:
                    continue

                ndbi_results.append({
                    "ward_id": ward_id,
                    "date": item.datetime.date(),
                    "scene_id": item.id,
                    "tile": (
                        "T43QHV"
                        if "T43QHV" in item.id
                        else "T44QKE"
                    ),
                    "ndbi_mean": float(np.mean(ndbi)),
                    "ndbi_median": float(np.median(ndbi)),
                    "ndbi_min": float(np.min(ndbi)),
                    "ndbi_max": float(np.max(ndbi)),
                    "valid_pixels": int(len(ndbi))
                })

ndbi_scene_df = pd.DataFrame(ndbi_results)

print("\nExtraction complete")
print("Rows:", len(ndbi_scene_df))
print("Unique wards:", ndbi_scene_df["ward_id"].nunique())
print("Unique scenes:", ndbi_scene_df["scene_id"].nunique())
print("Unique tiles:", ndbi_scene_df["tile"].nunique())
print("Missing median NDBI:", ndbi_scene_df["ndbi_median"].isna().sum())

Processing 1/19: S2C_MSIL2A_20250303T050741_R019_T43QHV_20250303T103811
Processing 2/19: S2C_MSIL2A_20250303T050741_R019_T44QKE_20250303T103811
Processing 3/19: S2B_MSIL2A_20250308T050659_R019_T43QHV_20250308T072151
Processing 4/19: S2B_MSIL2A_20250308T050659_R019_T44QKE_20250308T072151
Processing 5/19: S2C_MSIL2A_20250313T050711_R019_T43QHV_20250313T102208
Processing 6/19: S2C_MSIL2A_20250313T050711_R019_T44QKE_20250313T102208
Processing 7/19: S2A_MSIL2A_20250315T051241_R019_T43QHV_20250315T113317
Processing 8/19: S2A_MSIL2A_20250315T051241_R019_T44QKE_20250315T113317
Processing 9/19: S2B_MSIL2A_20250318T050659_R019_T43QHV_20250318T080831
Processing 10/19: S2B_MSIL2A_20250318T050659_R019_T44QKE_20250318T080831
Processing 11/19: S2B_MSIL2A_20250328T050659_R019_T43QHV_20250328T072008
Processing 12/19: S2B_MSIL2A_20250328T050659_R019_T44QKE_20250328T072008
Processing 13/19: S2B_MSIL2A_20250407T050649_R019_T43QHV_20250407T074451
Processing 14/19: S2B_MSIL2A_20250407T050649_R019_T44QKE_202

In [30]:
ward_ndbi_2025 = (
    ndbi_scene_df
    .groupby("ward_id", as_index=False)
    .agg(
        ndbi_2025=("ndbi_median", "median"),
        ndbi_mean_across_scenes=("ndbi_mean", "mean"),
        ndbi_min_observed=("ndbi_min", "min"),
        ndbi_max_observed=("ndbi_max", "max"),
        observations=("date", "count"),
        total_valid_pixels=("valid_pixels", "sum")
    )
)

print("Rows:", len(ward_ndbi_2025))
print("Unique wards:", ward_ndbi_2025["ward_id"].nunique())
print("Missing NDBI:", ward_ndbi_2025["ndbi_2025"].isna().sum())

print("\nNDBI summary:")
print(
    ward_ndbi_2025["ndbi_2025"].describe()
)

print("\nFirst 10 wards:")
print(
    ward_ndbi_2025.head(10)
)

Rows: 150
Unique wards: 150
Missing NDBI: 0

NDBI summary:
count    150.000000
mean       0.052501
std        0.021309
min       -0.094833
25%        0.041594
50%        0.053436
75%        0.066732
max        0.098108
Name: ndbi_2025, dtype: float64

First 10 wards:
   ward_id  ndbi_2025  ndbi_mean_across_scenes  ndbi_min_observed  \
0        1   0.040347                 0.034882          -0.376076   
1        2   0.040523                 0.036588          -0.261774   
2        3   0.062695                 0.055201          -0.364842   
3        4   0.045722                 0.044535          -0.254400   
4        5   0.065068                 0.061900          -0.320563   
5        6   0.036722                 0.026022          -0.425728   
6        7   0.036366                 0.027288          -0.307734   
7        8   0.034793                 0.026220          -0.350221   
8        9   0.045302                 0.029470          -0.370611   
9       10   0.053604                 0.04

In [31]:
heat_master = heat_master.merge(
    ward_ndbi_2025,
    on="ward_id",
    how="left",
    validate="one_to_one"
)

print("Rows:", len(heat_master))
print("Columns:", len(heat_master.columns))
print("Unique wards:", heat_master["ward_id"].nunique())

print("\nMissing values:")
print(
    heat_master[
        ["lst_2025_c", "ndvi_2025", "ndbi_2025"]
    ].isna().sum()
)

print("\nKey indicators:")
print(
    heat_master[
        ["ward_id", "lst_2025_c", "ndvi_2025", "ndbi_2025"]
    ].head(10)
)

Rows: 150
Columns: 22
Unique wards: 150

Missing values:
lst_2025_c    0
ndvi_2025     0
ndbi_2025     0
dtype: int64

Key indicators:
   ward_id  lst_2025_c  ndvi_2025  ndbi_2025
0        1   37.963271   0.148417   0.040347
1        2   37.864148   0.136789   0.040523
2        3   39.450109   0.143996   0.062695
3        4   38.506736   0.121354   0.045722
4        5   39.815837   0.134192   0.065068
5        6   38.510154   0.125300   0.036722
6        7   38.264056   0.131655   0.036366
7        8   38.238421   0.161975   0.034793
8        9   38.722071   0.106715   0.045302
9       10   40.270434   0.127327   0.053604


In [32]:
heat_master_geojson = os.path.join(
    processed_path,
    "hyderabad_ward_heat_environment_master_2025.geojson"
)

heat_master_csv = os.path.join(
    processed_path,
    "hyderabad_ward_heat_environment_master_2025.csv"
)

heat_master.to_file(
    heat_master_geojson,
    driver="GeoJSON"
)

heat_master.drop(
    columns="geometry"
).to_csv(
    heat_master_csv,
    index=False
)

print("GeoJSON saved:", heat_master_geojson)
print("CSV saved:", heat_master_csv)
print("Rows:", len(heat_master))
print("Columns:", len(heat_master.columns))

GeoJSON saved: /content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/hyderabad_ward_heat_environment_master_2025.geojson
CSV saved: /content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/hyderabad_ward_heat_environment_master_2025.csv
Rows: 150
Columns: 22


In [33]:
heat_master["heat_percentile"] = (
    heat_master["lst_2025_c"]
    .rank(pct=True)
)

heat_master["density_percentile"] = (
    heat_master["population_density_2011"]
    .rank(pct=True)
)

print("Missing heat percentile:", heat_master["heat_percentile"].isna().sum())
print("Missing density percentile:", heat_master["density_percentile"].isna().sum())

print("\nHeat percentile summary:")
print(heat_master["heat_percentile"].describe())

print("\nDensity percentile summary:")
print(heat_master["density_percentile"].describe())

print("\nWards with missing population density:")
print(
    heat_master.loc[
        heat_master["population_density_2011"].isna(),
        ["ward_id", "ward_name", "lst_2025_c"]
    ]
)

Missing heat percentile: 0
Missing density percentile: 2

Heat percentile summary:
count    150.000000
mean       0.503333
std        0.289632
min        0.006667
25%        0.255833
50%        0.503333
75%        0.754167
max        1.000000
Name: heat_percentile, dtype: float64

Density percentile summary:
count    148.000000
mean       0.503378
std        0.289649
min        0.006757
25%        0.255068
50%        0.503378
75%        0.751689
max        1.000000
Name: density_percentile, dtype: float64

Wards with missing population density:
     ward_id         ward_name  lst_2025_c
114      115  115-BALAJI NAGAR   37.898328
115      116       116-ALLAPUR   38.817776


In [34]:
heat_master["heat_density_category"] = pd.NA

valid_density = heat_master["density_percentile"].notna()

heat_master.loc[
    valid_density &
    (heat_master["heat_percentile"] >= 0.5) &
    (heat_master["density_percentile"] >= 0.5),
    "heat_density_category"
] = "High Heat / High Density"

heat_master.loc[
    valid_density &
    (heat_master["heat_percentile"] >= 0.5) &
    (heat_master["density_percentile"] < 0.5),
    "heat_density_category"
] = "High Heat / Lower Density"

heat_master.loc[
    valid_density &
    (heat_master["heat_percentile"] < 0.5) &
    (heat_master["density_percentile"] >= 0.5),
    "heat_density_category"
] = "Lower Heat / High Density"

heat_master.loc[
    valid_density &
    (heat_master["heat_percentile"] < 0.5) &
    (heat_master["density_percentile"] < 0.5),
    "heat_density_category"
] = "Lower Heat / Lower Density"

print("Category counts:")
print(
    heat_master["heat_density_category"]
    .value_counts(dropna=False)
)

print("\nCategory percentages:")
print(
    heat_master["heat_density_category"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

print("\nMissing-category wards:")
print(
    heat_master.loc[
        heat_master["heat_density_category"].isna(),
        ["ward_id", "ward_name", "lst_2025_c", "population_2011"]
    ]
)

Category counts:
heat_density_category
High Heat / High Density      44
Lower Heat / Lower Density    42
High Heat / Lower Density     31
Lower Heat / High Density     31
<NA>                           2
Name: count, dtype: int64

Category percentages:
heat_density_category
High Heat / High Density      29.33
Lower Heat / Lower Density    28.00
High Heat / Lower Density     20.67
Lower Heat / High Density     20.67
<NA>                           1.33
Name: proportion, dtype: float64

Missing-category wards:
     ward_id         ward_name  lst_2025_c  population_2011
114      115  115-BALAJI NAGAR   37.898328              NaN
115      116       116-ALLAPUR   38.817776              NaN


In [35]:
heat_master["heat_rank"] = (
    heat_master["lst_2025_c"]
    .rank(method="min", ascending=False)
    .astype(int)
)

heat_master["vegetation_rank"] = (
    heat_master["ndvi_2025"]
    .rank(method="min", ascending=False)
    .astype(int)
)

heat_master["built_up_intensity_rank"] = (
    heat_master["ndbi_2025"]
    .rank(method="min", ascending=False)
    .astype(int)
)

print("Heat rank range:",
      heat_master["heat_rank"].min(),
      "to",
      heat_master["heat_rank"].max())

print("Vegetation rank range:",
      heat_master["vegetation_rank"].min(),
      "to",
      heat_master["vegetation_rank"].max())

print("Built-up intensity rank range:",
      heat_master["built_up_intensity_rank"].min(),
      "to",
      heat_master["built_up_intensity_rank"].max())

print("\nHighest heat wards:")
print(
    heat_master[
        ["ward_id", "ward_name", "lst_2025_c", "heat_rank"]
    ]
    .sort_values("heat_rank")
    .head(10)
)

print("\nHighest built-up intensity wards:")
print(
    heat_master[
        ["ward_id", "ward_name", "ndbi_2025", "built_up_intensity_rank"]
    ]
    .sort_values("built_up_intensity_rank")
    .head(10)
)

print("\nHighest vegetation wards:")
print(
    heat_master[
        ["ward_id", "ward_name", "ndvi_2025", "vegetation_rank"]
    ]
    .sort_values("vegetation_rank")
    .head(10)
)

Heat rank range: 1 to 150
Vegetation rank range: 1 to 150
Built-up intensity rank range: 1 to 150

Highest heat wards:
     ward_id           ward_name  lst_2025_c  heat_rank
12        13      13-HAYATHNAGAR   41.719675          1
11        12      12-MANSOORABAD   41.446233          2
45        46        46-FALAKNUMA   40.834407          3
33        34  34-TALABCHANCHALAM   40.601982          4
13        14  14-B.N.REDDY NAGAR   40.434499          5
31        32     32-PATHER GATTI   40.349049          6
129      130   130-SUBHASH NAGAR   40.328540          7
9         10            10-UPPAL   40.270434          8
46        47  47-NAWABSAHEBKUNTA   40.137131          9
30        31       31-REIN BAZAR   40.123459         10

Highest built-up intensity wards:
     ward_id             ward_name  ndbi_2025  built_up_intensity_rank
33        34    34-TALABCHANCHALAM   0.098108                        1
45        46          46-FALAKNUMA   0.093976                        2
46        47    4

In [36]:
from scipy.stats import pearsonr, spearmanr

analysis_df = heat_master[
    [
        "ward_id",
        "lst_2025_c",
        "ndvi_2025",
        "ndbi_2025",
        "population_density_2011"
    ]
].copy()

pairs = [
    ("LST vs NDBI", "lst_2025_c", "ndbi_2025"),
    ("LST vs NDVI", "lst_2025_c", "ndvi_2025"),
    ("LST vs Population Density", "lst_2025_c", "population_density_2011")
]

results = []

for relationship, x_col, y_col in pairs:
    pair_df = analysis_df[[x_col, y_col]].dropna()

    pearson_r, pearson_p = pearsonr(
        pair_df[x_col],
        pair_df[y_col]
    )

    spearman_rho, spearman_p = spearmanr(
        pair_df[x_col],
        pair_df[y_col]
    )

    results.append({
        "relationship": relationship,
        "n_wards": len(pair_df),
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p
    })

correlation_results = pd.DataFrame(results)

print(correlation_results.to_string(index=False))


             relationship  n_wards  pearson_r    pearson_p  spearman_rho   spearman_p
              LST vs NDBI      150   0.794918 6.251098e-34      0.730676 2.619519e-26
              LST vs NDVI      150  -0.342358 1.800872e-05     -0.377353 1.934679e-06
LST vs Population Density      148   0.176062 3.231629e-02      0.162094 4.903670e-02


In [37]:
heat_master["heat_percentile"] = heat_master["lst_2025_c"].rank(pct=True)

heat_master["vegetation_percentile"] = heat_master["ndvi_2025"].rank(pct=True)

heat_master["built_up_percentile"] = heat_master["ndbi_2025"].rank(pct=True)

heat_master["density_percentile"] = heat_master["population_density_2011"].rank(pct=True)

heat_master["heat_rank"] = (
    heat_master["lst_2025_c"]
    .rank(method="min", ascending=False)
    .astype(int)
)

heat_master["vegetation_rank"] = (
    heat_master["ndvi_2025"]
    .rank(method="min", ascending=False)
    .astype(int)
)

heat_master["built_up_intensity_rank"] = (
    heat_master["ndbi_2025"]
    .rank(method="min", ascending=False)
    .astype(int)
)

heat_master["heat_level"] = pd.cut(
    heat_master["heat_percentile"],
    bins=[0, 0.25, 0.50, 0.75, 1.00],
    labels=[
        "Lower Heat",
        "Moderate Heat",
        "Higher Heat",
        "Highest Heat"
    ],
    include_lowest=True
)

heat_master["vegetation_level"] = pd.cut(
    heat_master["vegetation_percentile"],
    bins=[0, 0.25, 0.50, 0.75, 1.00],
    labels=[
        "Lower Vegetation",
        "Moderate Vegetation",
        "Higher Vegetation",
        "Highest Vegetation"
    ],
    include_lowest=True
)

heat_master["built_up_level"] = pd.cut(
    heat_master["built_up_percentile"],
    bins=[0, 0.25, 0.50, 0.75, 1.00],
    labels=[
        "Lower Built-up",
        "Moderate Built-up",
        "Higher Built-up",
        "Highest Built-up"
    ],
    include_lowest=True
)

heat_master["heat_density_category"] = pd.NA

valid_density = heat_master["density_percentile"].notna()

heat_master.loc[
    valid_density &
    (heat_master["heat_percentile"] >= 0.50) &
    (heat_master["density_percentile"] >= 0.50),
    "heat_density_category"
] = "High Heat / High Density"

heat_master.loc[
    valid_density &
    (heat_master["heat_percentile"] >= 0.50) &
    (heat_master["density_percentile"] < 0.50),
    "heat_density_category"
] = "High Heat / Lower Density"

heat_master.loc[
    valid_density &
    (heat_master["heat_percentile"] < 0.50) &
    (heat_master["density_percentile"] >= 0.50),
    "heat_density_category"
] = "Lower Heat / High Density"

heat_master.loc[
    valid_density &
    (heat_master["heat_percentile"] < 0.50) &
    (heat_master["density_percentile"] < 0.50),
    "heat_density_category"
] = "Lower Heat / Lower Density"

print("Rows:", len(heat_master))
print("Columns:", len(heat_master.columns))
print("Unique wards:", heat_master["ward_id"].nunique())

print("\nHeat levels:")
print(heat_master["heat_level"].value_counts(dropna=False).sort_index())

print("\nVegetation levels:")
print(heat_master["vegetation_level"].value_counts(dropna=False).sort_index())

print("\nBuilt-up levels:")
print(heat_master["built_up_level"].value_counts(dropna=False).sort_index())

print("\nHeat-density categories:")
print(
    heat_master["heat_density_category"]
    .value_counts(dropna=False)
)

Rows: 150
Columns: 33
Unique wards: 150

Heat levels:
heat_level
Lower Heat       37
Moderate Heat    38
Higher Heat      37
Highest Heat     38
Name: count, dtype: int64

Vegetation levels:
vegetation_level
Lower Vegetation       37
Moderate Vegetation    38
Higher Vegetation      37
Highest Vegetation     38
Name: count, dtype: int64

Built-up levels:
built_up_level
Lower Built-up       37
Moderate Built-up    38
Higher Built-up      37
Highest Built-up     38
Name: count, dtype: int64

Heat-density categories:
heat_density_category
High Heat / High Density      44
Lower Heat / Lower Density    42
High Heat / Lower Density     31
Lower Heat / High Density     31
<NA>                           2
Name: count, dtype: int64


In [39]:
final_columns = [
    "ward_id",
    "ward_name",
    "circle",
    "zone",
    "area_sq_km",
    "population_2011",
    "households_2011",
    "population_density_2011",
    "lst_2025_c",
    "ndvi_2025",
    "ndbi_2025",
    "heat_percentile",
    "vegetation_percentile",
    "built_up_percentile",
    "density_percentile",
    "heat_rank",
    "vegetation_rank",
    "built_up_intensity_rank",
    "heat_level",
    "vegetation_level",
    "built_up_level",
    "heat_density_category",
    "geometry"
]

final_master = heat_master[final_columns].copy()

final_master_csv = os.path.join(
    processed_path,
    "hyderabad_urban_heat_analytics_2025.csv"
)

final_master_geojson = os.path.join(
    processed_path,
    "hyderabad_urban_heat_analytics_2025.geojson"
)

final_master.drop(
    columns="geometry"
).to_csv(
    final_master_csv,
    index=False
)

final_master.to_file(
    final_master_geojson,
    driver="GeoJSON"
)

print("Final CSV:", final_master_csv)
print("Final GeoJSON:", final_master_geojson)
print("Rows:", len(final_master))
print("Columns:", len(final_master.columns))
print("Unique wards:", final_master["ward_id"].nunique())
print("Missing LST:", final_master["lst_2025_c"].isna().sum())
print("Missing NDVI:", final_master["ndvi_2025"].isna().sum())
print("Missing NDBI:", final_master["ndbi_2025"].isna().sum())
print("Missing population:", final_master["population_2011"].isna().sum())
print("Missing density:", final_master["population_density_2011"].isna().sum())

Final CSV: /content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/hyderabad_urban_heat_analytics_2025.csv
Final GeoJSON: /content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/hyderabad_urban_heat_analytics_2025.geojson
Rows: 150
Columns: 23
Unique wards: 150
Missing LST: 0
Missing NDVI: 0
Missing NDBI: 0
Missing population: 2
Missing density: 2


In [40]:
correlation_csv = os.path.join(
    processed_path,
    "hyderabad_heat_correlation_summary_2025.csv"
)

correlation_results.to_csv(
    correlation_csv,
    index=False
)

print("Correlation summary saved:", correlation_csv)
print("\nCorrelation summary:")
print(correlation_results.to_string(index=False))

Correlation summary saved: /content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/data/processed/hyderabad_heat_correlation_summary_2025.csv

Correlation summary:
             relationship  n_wards  pearson_r    pearson_p  spearman_rho   spearman_p
              LST vs NDBI      150   0.794918 6.251098e-34      0.730676 2.619519e-26
              LST vs NDVI      150  -0.342358 1.800872e-05     -0.377353 1.934679e-06
LST vs Population Density      148   0.176062 3.231629e-02      0.162094 4.903670e-02


In [41]:
print("FINAL DATASET QA")
print("-" * 30)

print("Rows:", len(final_master))
print("Columns:", len(final_master.columns))
print("Unique ward IDs:", final_master["ward_id"].nunique())
print(
    "Ward ID range:",
    final_master["ward_id"].min(),
    "to",
    final_master["ward_id"].max()
)

print("\nGeometry:")
print("CRS:", final_master.crs)
print("Null geometries:", final_master.geometry.isna().sum())
print("Empty geometries:", final_master.geometry.is_empty.sum())
print("Invalid geometries:", (~final_master.geometry.is_valid).sum())

print("\nSatellite indicators:")
print("Missing LST:", final_master["lst_2025_c"].isna().sum())
print("Missing NDVI:", final_master["ndvi_2025"].isna().sum())
print("Missing NDBI:", final_master["ndbi_2025"].isna().sum())

print("\nPopulation indicators:")
print("Missing population:", final_master["population_2011"].isna().sum())
print("Missing density:", final_master["population_density_2011"].isna().sum())

print("\nAnalytical categories:")
print(
    "Heat-density missing:",
    final_master["heat_density_category"].isna().sum()
)

print("\nDuplicate wards:", final_master["ward_id"].duplicated().sum())

FINAL DATASET QA
------------------------------
Rows: 150
Columns: 23
Unique ward IDs: 150
Ward ID range: 1 to 150

Geometry:
CRS: EPSG:4326
Null geometries: 0
Empty geometries: 0
Invalid geometries: 0

Satellite indicators:
Missing LST: 0
Missing NDVI: 0
Missing NDBI: 0

Population indicators:
Missing population: 2
Missing density: 2

Analytical categories:
Heat-density missing: 2

Duplicate wards: 0


In [42]:
data_dictionary = """
# Hyderabad Urban Heat Analytics — Data Dictionary

## Dataset

File: `hyderabad_urban_heat_analytics_2025.csv`

Coverage: 150 current GHMC wards

Reference year:
- Satellite indicators: 2025
- Population indicators: 2011 Census

## Ward and Geography Fields

| Field | Description |
|---|---|
| ward_id | Current GHMC ward identifier, ranging from 1 to 150. |
| ward_name | Current GHMC ward name. |
| circle | GHMC administrative circle. |
| zone | GHMC administrative zone. |
| area_sq_km | Current GHMC ward area in square kilometres. |
| geometry | Current GHMC ward polygon in EPSG:4326. |

## Population Fields

| Field | Description |
|---|---|
| population_2011 | Ward-level population from the 2011 Census. |
| households_2011 | Ward-level household count from the 2011 Census. |
| population_density_2011 | 2011 Census population divided by current GHMC ward area. This is a baseline approximation because current GHMC ward boundaries are not guaranteed to match the historical Census geography exactly. |

Population data is unavailable for wards 115 and 116 in the retrieved 2011 Census ward-level records. These values are retained as missing rather than estimated.

## Satellite Indicators

| Field | Description |
|---|---|
| lst_2025_c | Representative 2025 land surface temperature in degrees Celsius, calculated as the median of scene-level median LST values from selected hot-season Landsat scenes. |
| ndvi_2025 | Representative 2025 Normalized Difference Vegetation Index, calculated as the median of scene-level median NDVI values from selected Sentinel-2 scenes. |
| ndbi_2025 | Representative 2025 Normalized Difference Built-up Index, calculated as the median of scene-level median NDBI values from selected Sentinel-2 scenes. It represents built-up intensity, not built-up area percentage. |

LST represents land surface temperature rather than near-surface air temperature.

NDVI processing retained Sentinel-2 Scene Classification Layer classes 4 and 5, representing vegetation and bare soil.

NDBI was calculated using Sentinel-2 B08 NIR and B11 SWIR bands on the 20 m B11 grid.

## Percentile Fields

| Field | Description |
|---|---|
| heat_percentile | Relative percentile position of each ward based on 2025 LST. |
| vegetation_percentile | Relative percentile position based on 2025 NDVI. |
| built_up_percentile | Relative percentile position based on 2025 NDBI. |
| density_percentile | Relative percentile position based on 2011 population density where available. |

## Ranking Fields

| Field | Description |
|---|---|
| heat_rank | Rank based on 2025 LST, with rank 1 representing the highest value. |
| vegetation_rank | Rank based on 2025 NDVI, with rank 1 representing the highest value. |
| built_up_intensity_rank | Rank based on 2025 NDBI, with rank 1 representing the highest value. |

## Analytical Categories

| Field | Description |
|---|---|
| heat_level | Four percentile-based groups: Lower Heat, Moderate Heat, Higher Heat, Highest Heat. |
| vegetation_level | Four percentile-based groups: Lower Vegetation, Moderate Vegetation, Higher Vegetation, Highest Vegetation. |
| built_up_level | Four percentile-based groups: Lower Built-up, Moderate Built-up, Higher Built-up, Highest Built-up. |
| heat_density_category | Combines heat and population-density percentile groups into four categories. Missing where population density is unavailable. |

## Interpretation Notes

Correlation results describe statistical associations between ward-level indicators and should not be interpreted as evidence of causation.

The analysis is based on current GHMC ward boundaries and should not be interpreted as a historical ward-boundary comparison.

Satellite-derived LST, NDVI and NDBI represent remotely sensed environmental indicators and are subject to spatial, temporal and atmospheric limitations.
"""

data_dictionary_path = os.path.join(
    project_path,
    "docs",
    "data_dictionary.md"
)

os.makedirs(
    os.path.dirname(data_dictionary_path),
    exist_ok=True
)

with open(
    data_dictionary_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(data_dictionary)

print("Data dictionary saved:")
print(data_dictionary_path)

Data dictionary saved:
/content/drive/MyDrive/Hyderabad_Urban_Heat_Analytics/docs/data_dictionary.md
